In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import torch
import pickle


In [ ]:
UHM_DATASET_DIR = Path("/data/UHM_generated_data")

In [ ]:
DATASET_DIR = Path("../data/faces")

In [ ]:
all_files = list(UHM_DATASET_DIR.glob("*.ply"))

In [ ]:
RANDOM_SEED = 42

In [ ]:
np.random.seed(RANDOM_SEED)

In [ ]:
N_SAMPLES = 500

In [ ]:
random_sample = np.random.choice(all_files, size=N_SAMPLES, replace=False)

In [ ]:
random_sample

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
def generate_random_view(
    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9
) -> np.ndarray:

    center = pcd_points.mean(axis=0)
    extent_y = np.ptp(pts[:, 1])
    extent_z = np.ptp(pts[:, 2])
    height = extent_y * plane_height_factor
    depth = extent_z * plane_depth_factor

    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)
    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])

    x_angle = rng.uniform(0, 2 * np.pi)
    y_angle = rng.uniform(0, 2 * np.pi)
    z_angle = rng.uniform(0, 2 * np.pi)
    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])
    plane.rotate(R_plane, center=center)

    initial_normal = np.array([1.0, 0.0, 0.0])
    rotated_normal = R_plane @ initial_normal
    rotated_normal /= np.linalg.norm(rotated_normal)

    vecs = pcd_points - center[np.newaxis, :]
    signed = vecs.dot(rotated_normal)
    side = rng.choice([1, -1])
    mask_side = (signed * side) > side_tol
    selected_pts = pcd_points[mask_side]
    orig_indices = np.nonzero(mask_side)[0]

    down_mask_bool = rng.choice([False, True], size=selected_pts.shape[0],
                                p=[1 - downsample_p, downsample_p])
    downsampled_pts = selected_pts[down_mask_bool]
    kept_indices = orig_indices[down_mask_bool]

    angles = rng.uniform(0, 2 * np.pi, size=3)
    R = o3d.geometry.get_rotation_matrix_from_xyz(angles)
    t = rng.uniform(-1.0, 1.0, size=3)
    transformed_points = (R @ downsampled_pts.T).T + t

    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4, dtype=float)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv

    print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((transformed_points, np.ones((transformed_points.shape[0], 1)))) @ T_inv.T)[:, :3] - pts[kept_indices])}")

    return transformed_points, R_inv, t_inv, plane, kept_indices

In [ ]:
random_file = random_sample[0]

In [ ]:
pcd = o3d.io.read_point_cloud(str(random_file))
pcd.colors = o3d.utility.Vector3dVector(np.ones((len(pcd.points), 3)) * 0.5)
pts = np.asarray(pcd.points)

In [ ]:
view_points, view_r, view_t, view_plane, kept_indices = generate_random_view(pts)

In [ ]:
view_r.dtype

In [ ]:
view_pcd = o3d.geometry.PointCloud()
view_pcd.points = o3d.utility.Vector3dVector(view_points)

In [ ]:
o3d.visualization.draw_plotly([view_pcd, view_plane, pcd])

In [ ]:
T = np.eye(4, dtype=np.float64)
T[:3, :3] = view_r
T[:3, 3] = view_t

In [ ]:
transformed_view = view_pcd.transform(T)

In [ ]:
o3d.visualization.draw_plotly([transformed_view, view_plane])

In [ ]:
def build_metadata_dict(scene_name, pcd0_path, pcd1_path, R, t, frag_id1, kept_indices):
    metadata = {
        "overlap": 0,
        "pcd0": str(pcd0_path),
        "pcd1": str(pcd1_path),
        "rotation": R,
        "translation": t,
        "scene_name": scene_name,
        "frag_id0": 0,
        "frag_id1": frag_id1,
        "kept_indices": kept_indices,
    }
    return metadata

In [ ]:
FULL_PC_NAME =  "full_face.pth"

In [ ]:
def process_file(file_path: Path, n_views=10, folder="train", save=True):
    pcd = o3d.io.read_point_cloud(str(file_path))
    pts = np.asarray(pcd.points)

    print(f"Processing file: {file_path.stem} with {pts.shape[0]} points. Data type: {pts.dtype}")

    subject_path = DATASET_DIR / "data" / folder / file_path.stem
    if save:
        subject_path.mkdir(parents=True, exist_ok=True)
        torch.save(pts, subject_path / FULL_PC_NAME)
    metadata_list = []

    for i in range(n_views):
        transformed_points, R_inv, t_inv, _, kept_indices = generate_random_view(pts)

        metadata = build_metadata_dict(
            scene_name=file_path.stem,
            pcd0_path=Path(folder) / file_path.stem / FULL_PC_NAME,
            pcd1_path=Path(folder) / file_path.stem / f"view_{i+1}.pth",
            R=R_inv,
            t=t_inv,
            frag_id1=i + 1,
            kept_indices=kept_indices
        )
        metadata_list.append(metadata)
        if save:    
            torch.save(transformed_points, subject_path / f"view_{i+1}.pth")

    return metadata_list

In [ ]:
train_size = int(np.round(0.8 * len(random_sample)))
val_size = int(np.round(0.1 * len(random_sample)))
test_size = len(random_sample) - train_size - val_size

In [ ]:
train_size, val_size, test_size

In [ ]:
train_metadata = []
for file_path in random_sample[:train_size]:
    metadata_list = process_file(file_path, n_views=10, folder="train")
    train_metadata.extend(metadata_list)
val_metadata = []
for file_path in random_sample[train_size:train_size+val_size]:
    metadata_list = process_file(file_path, n_views=10, folder="train")
    val_metadata.extend(metadata_list)

In [ ]:
test_metadata = []
for file_path in random_sample[train_size+val_size:]:
    metadata_list = process_file(file_path, n_views=1, folder="test")
    test_metadata.extend(metadata_list)

In [ ]:
METADATA_DIR = DATASET_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
with open(METADATA_DIR / "train.pkl", "wb") as f:
    pickle.dump(train_metadata, f)

In [ ]:
with open(METADATA_DIR / "val.pkl", "wb") as f:
    pickle.dump(val_metadata, f)

In [ ]:
demo_folder = DATASET_DIR / "demo"
demo_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
for ex_id, example in enumerate(test_metadata):
    example_ref = torch.load(DATASET_DIR / "data" / example["pcd0"])
    example_src = torch.load(DATASET_DIR / "data" / example["pcd1"])
    np.save(demo_folder / f"ref_{ex_id}.npy", example_ref)
    np.save(demo_folder / f"src_{ex_id}.npy", example_src)
    rot = example["rotation"]
    t = example["translation"]
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = rot
    T[:3, 3] = t
    np.save(demo_folder / f"gt_{ex_id}.npy", T)


### Inspect Test Data

In [ ]:
#test_sample = test_metadata[np.random.randint(len(test_metadata))]
test_sample = test_metadata[0]

In [ ]:
test_sample

In [ ]:
test_src = torch.load(DATASET_DIR / "data" / test_sample["pcd1"])
test_ref = torch.load(DATASET_DIR / "data" / test_sample["pcd0"])

In [ ]:
file_path = Path("/data/UHM_generated_data") / f"{test_sample['scene_name']}.ply" 

In [ ]:
ref_original = pcd = o3d.io.read_point_cloud(str(file_path))

In [ ]:
test_ref.shape

In [ ]:
np.asarray(ref_original.points).shape

In [ ]:
np.mean(np.linalg.norm(np.asarray(ref_original.points) - test_ref, axis=1))

In [ ]:
src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0])
ref_pcd = o3d.geometry.PointCloud()
ref_pcd.points = o3d.utility.Vector3dVector(test_ref)
ref_pcd.paint_uniform_color([0.0, 1.0, 0.0])


In [ ]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [ ]:
t = test_sample["translation"]
rot = test_sample["rotation"]
T = np.eye(4, dtype=np.float64)
T[:3, :3] = rot
T[:3, 3] = t

In [ ]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd.transform(T)])

In [ ]:
kept_indices = test_sample["kept_indices"]

In [ ]:
r_errors = (np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices]

In [ ]:
np.linalg.norm(r_errors, axis=1).max()

In [ ]:
print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices])}")


In [ ]:
error = np.asarray(ref_pcd.points)[kept_indices] - np.asarray(src_pcd.points)

In [ ]:
np.linalg.norm(error, axis=1).mean()

In [ ]:
np.sum(np.linalg.norm(error, axis=1))